# Lab03 Colab Runner

This notebook clones your GitHub repo, installs dependencies, builds the healthcare GraphRAG artifacts, runs GNN training, evaluates retrieval, and can launch the Gradio app.

Use this for preprocessing and validation on Colab, then deploy the final repo to Hugging Face Spaces.

## Before you run

1. Push this project to GitHub.
2. Replace the placeholder repo URL in the next cell.
3. If your repo is private, set `GITHUB_TOKEN` first or use a personal access token in the clone URL.

Notes:
- `llama-cpp-python` may take a while to build on Colab.
- The GGUF generation model will be downloaded only when you actually use the app.
- The build step can take a long time if you keep the default dataset size.

In [ ]:
# Replace these values before running.
GITHUB_REPO_URL = "https://github.com/your-username/your-lab03-repo.git"
GITHUB_BRANCH = "main"
REPO_DIR = "lab03_repo"

# Optional: reduce runtime for quick smoke tests.
MAX_RECORDS = 400
GNN_EPOCHS = 20
EVAL_LIMIT = 100

# Final Space deployment should usually use larger values.
# Example stronger run:
# MAX_RECORDS = 1500
# GNN_EPOCHS = 60
# EVAL_LIMIT = 200

In [ ]:
!apt-get -qq update
!apt-get -qq install -y build-essential cmake ninja-build

In [ ]:
!rm -rf {REPO_DIR}
!git clone --branch {GITHUB_BRANCH} {GITHUB_REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git rev-parse --short HEAD

In [ ]:
!python -m pip install --upgrade pip setuptools wheel
!python -m pip install -r requirements.txt

In [ ]:
import os

os.environ["MAX_RECORDS"] = str(MAX_RECORDS)
os.environ["GNN_EPOCHS"] = str(GNN_EPOCHS)
os.environ["EVAL_LIMIT"] = str(EVAL_LIMIT)

print({
    "MAX_RECORDS": os.environ["MAX_RECORDS"],
    "GNN_EPOCHS": os.environ["GNN_EPOCHS"],
    "EVAL_LIMIT": os.environ["EVAL_LIMIT"],
})

In [ ]:
!python scripts/build_all.py

In [ ]:
import json
from pathlib import Path

metrics_path = Path("evaluation/retrieval_metrics.json")
print("metrics file exists:", metrics_path.exists())
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    print(json.dumps(metrics, indent=2))

print("\nartifacts:")
!find artifacts -maxdepth 1 -type f | sort

print("\nstorage:")
!find storage -maxdepth 2 -type f | sort | head -n 50

## Optional: launch the Gradio app in Colab

Run the next cell only if you want to interact with the app from Colab.

Important:
- The first real query may trigger GGUF model download.
- This is much heavier than the preprocessing pipeline.
- If you only need artifacts for Hugging Face Spaces, you can skip this.

In [ ]:
from app import demo

demo.launch(share=True, debug=True)

## Optional: save generated artifacts back to GitHub

If you want your Hugging Face Space to start without rebuilding everything, commit these generated folders:

- `artifacts/`
- `storage/`
- `evaluation/`

You can use the next cell after setting your Git identity and authentication.

In [ ]:
# Optional only.
# !git config user.name "Your Name"
# !git config user.email "you@example.com"
# !git status
# !git add artifacts storage evaluation
# !git commit -m "Add prebuilt Lab03 artifacts from Colab"
# !git push origin {GITHUB_BRANCH}